# League of Legends

## Librairies

In [1]:
import sys
sys.path.append("../")

import requests
import time
import re
from datetime import datetime
import os
from dotenv import load_dotenv
import pandas as pd
import json
from src.keys_data import KeysData
load_dotenv()

True

## Extraction

In [2]:
api_key = os.getenv("API_KEY")
# Limit
# 20 requests evry 1 seconds
# 100 requests every 2 minutes

class ExtractPlayerData:

    def get_puuid(self, gameName : str, tagLine : str):
        """Get account by riot id  
        Returns :
        - puuid : Encrypted PUUID. Exact length of 78 characters.
        - gameName : Name of the account
        - tagLine : Indicatif of the account
        """
        api_url = f"https://europe.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp


    def get_summoner_info(self, puuid : str):
        """Get a summoner general information by PUUID  
        Returns :
        - profileiconId : ID of the summoner icon associated with the summoner.
        - revisionDate : last modification (events : profile icon change, playing tuto, finishing a game, summoner name change).
        - puuid : Encrypted PUUID. Exact length of 78 characters.
        - summonerLevel : Summoner level associated with the summoner.
        """

        api_url = f"https://euw1.api.riotgames.com/lol/summoner/v4/summoners/by-puuid/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
    

    def get_last_games_ids(self, puuid :str, start : int = 0, count : int = 20):
        """Get a list of match ids by puuid
        Returns :
        - List : List of last matches
        """

        api_url = f"https://europe.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start={start}&count={count}&api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp


    def get_all_games_ids(self, puuid: str, count_per_request: int = 100, delay: float = 1.5) -> list:
        """
        Récupère l'ensemble des IDs de parties d'un joueur via pagination.
        
        Args:
            puuid: Identifiant unique du joueur
            count_per_request: Nombre de résultats par requête (max 100 selon l'API Riot)
            delay: Délai en secondes entre chaque requête (adapter selon votre clé API)
        
        Returns:
            List[str]: Liste complète des IDs de toutes les parties jouées
        """
        all_match_ids = []
        start = 0

        while True:
            api_url = (
                f"https://europe.api.riotgames.com/lol/match/v5/matches/by-puuid/"
                f"{puuid}/ids?start={start}&count={count_per_request}&api_key={api_key}"
            )

            try:
                response = requests.get(api_url)

                # Gestion du rate limiting (429)
                if response.status_code == 429:
                    retry_after = int(response.headers.get("Retry-After", 10))
                    print(f"\t-Rate limit atteint. Attente de {retry_after}s...")
                    time.sleep(retry_after)
                    continue  # Rejoue la même requête

                response.raise_for_status()
                batch = response.json()

            except requests.exceptions.RequestException as e:
                print(f"\t-Erreur lors de la requête (start={start}) : {e}")
                break

            if not batch:
                break  # Plus de résultats, pagination terminée

            all_match_ids.extend(batch)
            print(f"\t- {len(all_match_ids)} parties récupérées...")

            # Si le batch est incomplet, c'est la dernière page
            if len(batch) < count_per_request:
                break

            start += count_per_request
            time.sleep(delay)  # Respect du rate limit entre les requêtes

        return all_match_ids


    def get_game_data(self, game_id : str):
        """Get all informations of a match by match id"""

        api_url = f"https://europe.api.riotgames.com/lol/match/v5/matches/{game_id}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
    

    def get_champion_mastery(self, puuid : str):
        """Get all champion mastery entries sorted by number of champion points descending."""

        api_url = f"https://euw1.api.riotgames.com/lol/champion-mastery/v4/champion-masteries/by-puuid/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
    

    def get_gamer_queue(self, puuid : str):
        """Get league entries in all queues for a given puuid"""
    
        api_url = f"https://euw1.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
    

    def get_challenges_info(self, puuid : str):
        """Returns player information with list of all progressed challenges (REST)"""

        api_url = f"https://euw1.api.riotgames.com/lol/challenges/v1/player-data/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
    
    
    def get_challenges_description(self, challenge_id : int):
        """Returns the description of a challenge"""

        api_url = f"https://euw1.api.riotgames.com/lol/challenges/v1/challenges/{challenge_id}/config?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp

In [3]:
class TransformPlayerData:

    def _dict_to_list_dict(self, dico : dict) -> list[dict]:
        """Ajoute un dico à une liste"""
        return [dico]
    
    def to_dataframe(self, data) -> pd.DataFrame:
        """Convertie des données de forme dict ou list[dict] en dataframe"""
        if type(data) == dict:
            data = self._dict_to_list_dict(data)
        return pd.DataFrame(data)
    
    def drop_col(self, df : pd.DataFrame, cols : list) -> pd.DataFrame:
        """Enlève des colonnes du dataframe"""
        return df.drop(columns=cols)

    def rename_df_cols(self, df : pd.DataFrame, renamer : dict) -> pd.DataFrame:
        """Rename des colonnes avec un dico {old_col_name : new_col_name}"""
        return df.rename(columns=renamer)
    
    def format_timestamp(self, df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
        """Formatage des colonnes listées en timestamp au format yyyy-mm-dd hh:mm:ss"""
        for col in cols:
            df[col] = pd.to_datetime(df[col], unit="ms").dt.floor("s")
        return df
    
    def add_puuid(self, df : pd.DataFrame, puuid : str) -> pd.DataFrame:
        """Ajoute le puuid du joueur à un dataframe"""
        df['puuid'] = puuid
        return df
    
    def map_catg_with_index(self, df : pd.DataFrame, col_to_map : str, name_map :str, mapper : dict) -> pd.DataFrame:
        """Ajoue une nouvelle colonne en mappant avec un dico de correspondance"""
        df[name_map] = df[col_to_map].map(mapper)
        return df
    
    def drill_down_dico(self, dico: dict, **kwargs):
        result = dico
        for key, value in kwargs.items():
            if isinstance(result, dict) and key in result:
                result = result[key]
            else:
                raise KeyError(f"Key '{key}' not found in dict")
        return result
    
    def add_key(self, dico : dict, new_key_val : dict):
        """Ajoute des pairs {clé : valeur} à un dictionnaire"""
        dico.update(new_key_val)
        return dico
    
    def clean_text(self, txt : str, exp : str, sub : str):
        """Remplace des éléments de texte spécifiques"""
        txt = re.sub(exp, sub, txt)
        return txt
    
    def merge_df(self, df1 : pd.DataFrame, df2 : pd.DataFrame, left_key : list[str], rigth_key : list[str], how : str = 'inner'):
        """Execute une jointure en deux dataframes"""
        df_merge = pd.merge(df1, df2, how=how, left_on=left_key, right_on=rigth_key)
        return df_merge
    
    def sec_to_minsec(self, secondes : int):
        """Convertie des secondes en format minute:secondes"""
        minu = secondes//60
        sec = secondes%60
        txt = f"{minu:02d}:{sec:02d}"
        return txt
    
    def add_underscore(self, col_names: list[str]) -> list[str]:
        """Convertit les noms de colonnes en snake_case (gestion avancée des majuscules)."""
        def camel_to_snake(name: str) -> str:
            s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name) # Insère _ avant les majuscules précédées de minuscules ou chiffres
            return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower() # Insère _ avant les majuscules suivies de minuscules
        
        return [camel_to_snake(name) for name in col_names]
    
    def parse_perks(perks: dict) -> dict:
        """
        Transforme le bloc 'perks' de l'API Riot Match v5
        en un dictionnaire plat, prêt à intégrer un DataFrame.
        """
        styles = perks["styles"]
        primary = next(s for s in styles if s["description"] == "primaryStyle")
        sub = next(s for s in styles if s["description"] == "subStyle")

        return {
            "id_tree_primary":    primary["style"],
            "id_primary_1":       primary["selections"][0]["perk"],
            "id_primary_2":       primary["selections"][1]["perk"],
            "id_primary_3":       primary["selections"][2]["perk"],
            "id_primary_4":       primary["selections"][3]["perk"],               
            "id_tree_sub":        sub["style"],
            "id_sub_1":           sub["selections"][0]["perk"],
            "id_sub_2":           sub["selections"][1]["perk"]
        }
    
    # def select_columns(self, df : pd.DataFrame, keys_words : list[str]) -> pd.DataFrame:
    #     """Capture les colonnes en fonction de mots clés"""
    #     keys_words += ['puuid', 'game_id']
    #     selected_cols = [x for x in df.columns if any(keyword in x.lower() for keyword in keys_words)]
    #     df_new = df[selected_cols]
    #     print(df_new.columns)
    #     return df_new
    
    def select_columns(self, df_base : pd.DataFrame, champs : list[str]) -> pd.DataFrame:
        """Capture les colonnes en fonction d'une liste"""
        df_new = df_base[champs]
        return df_new

In [ ]:
class Pipelines:

    def __init__(self):
        self.extract = ExtractPlayerData()
        self.transfo = TransformPlayerData()
        self.keys_data = KeysData()


    def pipeline_account(self, gameName : str, tagLine : str):
        """Récupération et traitement des données de compte"""

        print(f"- Récupération du puuid du joueur {gameName} (#{tagLine}).")
        df_account = self.extract.get_puuid(gameName=gameName, tagLine=tagLine)
        puuid = df_account['puuid']
        df_account = self.transfo.to_dataframe(data=df_account)
        df_account = self.transfo.rename_df_cols(df_account, renamer = {'gameName':'game_name', 'tagLine':'tag_line'})
        return df_account, puuid


    def pipeline_summoner(self, puuid : str) -> pd.DataFrame:
        """Récupération et traitement des données du summoner"""

        print("- Récupération des données du summoner du joueur.")
        df_summoner = self.extract.get_summoner_info(puuid=puuid)
        df_summoner = self.transfo.to_dataframe(data=df_summoner)
        df_summoner = self.transfo.rename_df_cols(df_summoner, renamer = {'profileIconId':'profile_icon_id', 'revisionDate':'last_modif', 'summonerLevel':'summoner_level'})
        df_summoner = self.transfo.format_timestamp(df_summoner, cols=['last_modif'])
        return df_summoner


    def pipeline_challenges(self, puuid : str) -> pd.DataFrame:
        """Récupération et traitement des données de challenges"""

        print("- Récupération des données d'avancement de challenges du joueur.")
        df_challenges = self.extract.get_challenges_info(puuid=puuid)['challenges']
        df_challenges = self.transfo.to_dataframe(df_challenges)
        df_challenges = self.transfo.format_timestamp(df_challenges, cols = ["achievedTime"])
        df_challenges = self.transfo.drop_col(df_challenges, cols=['position','playersInLevel'])
        df_challenges = self.transfo.rename_df_cols(df_challenges, renamer={'challengeId':'challenge_id', 'achievedTime':'achieved_time'})
        df_challenges = self.transfo.map_catg_with_index(df_challenges, col_to_map='level', name_map='level_index', 
                                                    mapper={'NONE' : 0, 'IRON' : 1, 'BRONZE' : 2, 'SILVER' : 3, 'GOLD' : 4, 
                                                            'PLATINUM' : 5, 'DIAMOND' : 6, 'MASTER' : 7, 'GRANDMASTER' : 8, 'CHALLENGER' : 9})
        all_challenge = []
        cnt = 0
        for id in df_challenges['challenge_id']:
            resp = self.extract.get_challenges_description(challenge_id=id)
            chall_id = self.transfo.drill_down_dico(dico=resp, id='id')
            chall_desc = self.transfo.drill_down_dico(dico=resp, localizedNames="localizedNames", fr_FR="fr_FR")
            chall_desc = self.transfo.add_key(dico=chall_desc, new_key_val={'challenge_id':chall_id})
            all_challenge.append(chall_desc)
            cnt += 1
            print(f"\t- Requete challenges envoyées : {cnt}", end='\r')
            time.sleep(1.5)
        
        all_challenge = self.transfo.to_dataframe(all_challenge)
        all_challenge = self.transfo.drop_col(all_challenge, cols=['shortDescription'])
        all_challenge['description'] = [self.transfo.clean_text(txt=x, exp=r"\xa0", sub="") for x in all_challenge['description']]
        all_challenge['name'] = [self.transfo.clean_text(txt=x, exp=r"\xa0", sub="") for x in all_challenge['name']]
        
        df = self.transfo.merge_df(df1=df_challenges, df2=all_challenge, left_key=['challenge_id'], rigth_key=['challenge_id'], how='left')
        df = self.transfo.add_puuid(df, puuid=puuid)
        return df


    def pipeline_champion_mastery(self, puuid : str) -> pd.DataFrame:
        """Récupération et traitement des données de maitrise des champions"""

        print("- Récupération des données de maîtrise de champions du joueur.")
        df_champion_mastery = self.extract.get_champion_mastery(puuid=puuid)
        df_champion_mastery = self.transfo.to_dataframe(df_champion_mastery)
        df_champion_mastery = self.transfo.drop_col(df_champion_mastery, cols=['championPointsSinceLastLevel', 'markRequiredForNextLevel',
                                                                        'tokensEarned', 'championSeasonMilestone', 'milestoneGrades', 
                                                                        'nextSeasonMilestone'])
        df_champion_mastery = self.transfo.format_timestamp(df_champion_mastery, cols = ['lastPlayTime'])
        df_champion_mastery = self.transfo.rename_df_cols(df_champion_mastery, renamer={'championId':'champ_id', 'championLevel':'champ_level',
                                                                                'championPoints':'champ_points', 'lastPlayTime':'last_time_played',
                                                                                'championPointsUntilNextLevel':'points_to_next_level'})
        return df_champion_mastery
    
    
    def pipeline_queue(self, puuid : str) -> pd.DataFrame:
        """Récupération et traitement des données de ranked"""

        print("- Récupération des données de queue (ranked)")
        df_queue = self.extract.get_gamer_queue(puuid=puuid)
        df_queue = self.transfo.to_dataframe(df_queue)
        df_queue = self.transfo.drop_col(df_queue, cols=['veteran', 'inactive', 'freshBlood', 'hotStreak'])
        df_queue = self.transfo.rename_df_cols(df_queue, renamer={'leagueId':'league_id', 'queueType':'queue_type', 'leaguePoints':'league_points'})
        return df_queue
    

    def pipeline_games(self, puuid : str):
        """Récupération et traitement des données de parties"""

        print("- Récupération des données de parties de jeux.")

        df_matchs_info = []
        df_relation_idgame_puuid = []

        list_id_games = self.extract.get_all_games_ids(puuid=puuid)

        for id in list_id_games:
            
            game = self.extract.get_game_data(game_id=id)

            game_info = game['info']
            game_info = {k:v for k,v in game_info.items() if k in self.keys_data.keys_match_info}
            game_info.update({'match_id' : id})

            df_matchs_info.append(game_info)
            
            game_id = game_info['gameId']
            participants = game['info']['participants']

            for participant in participants:
            
                puuid = participant['puuid']

                participant_data = participant
                participant_data = {k:v for k,v in participant_data.items() if k not in self.keys_data.keys_to_reject}
                participant_data.update({'game_id' : game_id})

                participant_challenge = participant['challenges']
                participant_challenge = {k:v for k,v in participant_challenge.items() if k not in self.keys_data.keys_to_reject_challenges}
                participant_challenge.update({'game_id' : game_id, 'puuid' : puuid})

                participant_perks = participant['perks']
                participant_perks = self.transfo.parse_perks(participant_perks)
                participant_perks.update({'game_id' : game_id, 'puuid' : puuid})
            
                df_relation_idgame_puuid.append({'game_id':id, 'puuid':puuid})

        df_matchs_info = self.transfo.to_dataframe(df_matchs_info)
        df_matchs_info.columns = self.transfo.add_underscore(col_names = df_matchs_info.columns)

        return df_matchs_info

In [ ]:
def pipeline_new_player(gameName : str, tagLine : str):
    """Insérer un nouveau joueur en base."""
    
    pipe = Pipelines()

    df_account, puuid = pipe.pipeline_account(gameName=gameName, tagLine=tagLine)
    df_summoner = pipe.pipeline_summoner(puuid=puuid)
    df_challenges = pipe.pipeline_challenges(puuid=puuid)
    df_champion_mastery = pipe.pipeline_champion_mastery(puuid=puuid)
    df_queue = pipe.pipeline_queue(puuid=puuid)

    return df_account, df_summoner, df_challenges, df_champion_mastery, df_queue

In [ ]:
df_account, df_summoner, df_challenges, df_champion_mastery, df_queue = pipeline_new_player(gameName='SaduSuricate5900', tagLine='47895')
df_challenges.to_csv(r"C:\Users\najim\Documents\Projets\LeagueOfLegends\data\clean\df_challenges.csv", index=False, encoding='utf-8', header=True)

In [ ]:
display(df_account)
display(df_summoner)
display(df_challenges)
display(df_champion_mastery)
display(df_queue)

In [ ]:
matches = ExtractPlayerData().get_last_games_ids(puuid = "uuhMlNBPDT3pLnQaGe2bXJ1CrxIuDxFD7dgcYKmdSnPL8sQ-TcgUDYLLjTLoZ94Lk6o-pu0ZtNRiUQ")
print("Ids des derniers matches :\n", matches)

In [ ]:
def write_json(data):
    with open(r'C:\Users\najim\Documents\Projets\LeagueOfLegends\data\raw\game_data.json', 'w', encoding='utf-8') as fichier:
    # indent=4 ajoute de l'indentation pour la lisibilité
        json.dump(data, fichier, indent=4, ensure_ascii=False)

In [3]:
def read_json(path):
    # Lire un fichier JSON
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [4]:
def treatgamedata():

    keys = KeysData()
    transfo = TransformPlayerData()

    def parse_perks(perks: dict) -> dict:
        """
        Transforme le bloc 'perks' de l'API Riot Match v5
        en un dictionnaire plat, prêt à intégrer un DataFrame.
        """
        styles = perks["styles"]
        primary = next(s for s in styles if s["description"] == "primaryStyle")
        sub = next(s for s in styles if s["description"] == "subStyle")

        return {
            # Voie principale
            "id_tree_primary":    primary["style"],
            "id_primary_1":      primary["selections"][0]["perk"],
            "id_primary_2":       primary["selections"][1]["perk"],
            "id_primary_3":       primary["selections"][2]["perk"],
            "id_primary_4":       primary["selections"][3]["perk"],               

            # Voie secondaire
            "id_tree_sub":        sub["style"],
            "id_sub_1":           sub["selections"][0]["perk"],
            "id_sub_2":           sub["selections"][1]["perk"]
        }

    match = read_json(r"C:\Users\najim\Documents\Projets\LeagueOfLegends\data\raw\game_data.json")

    df_participant_data = []
    df_participant_challenge = []
    df_participant_perks = []
    df_game_info = []
    df_relation_idgame_puuid = []

    id = match['metadata']['matchId']
    
    game_info = match['info']
    game_info = {k:v for k,v in game_info.items() if k in keys.keys_match_info}
    game_info.update({'match_id' : id})
    df_game_info.append(game_info)
    
    participants = match['info']['participants']

    for participant in participants:
        
        puuid = participant['puuid']

        participant_data = participant
        participant_data = {k:v for k,v in participant_data.items() if k not in keys.keys_to_reject}
        participant_data.update({'game_id' : game_info['gameId']})
        df_participant_data.append(participant_data)

        participant_challenge = participant['challenges']
        participant_challenge = {k:v for k,v in participant_challenge.items() if k not in keys.keys_to_reject_challenges}
        participant_challenge.update({'game_id' : game_info['gameId'], 'puuid' : puuid})
        df_participant_challenge.append(participant_challenge)

        participant_perks = participant['perks']
        participant_perks = parse_perks(participant_perks)
        participant_perks.update({'game_id' : game_info['gameId'], 'puuid' : puuid})
        df_participant_perks.append(participant_perks)

        print(participant_data)
        print(participant_challenge)
        print(participant_perks)
        print("\n")
        df_relation_idgame_puuid.append({'game_id':id, 'puuid':puuid})
    

    df_participant_data = pd.DataFrame(df_participant_data)
    df_participant_challenge = pd.DataFrame(df_participant_challenge)

    for i in df_participant_data.columns:
        for j in df_participant_challenge.columns:
            if i == j:
                print(f"Doublon cols : {i} / {j}")


    df_all_game_data = transfo.merge_df(df_participant_data, df_participant_challenge,
                                        left_key=['puuid', 'game_id'],
                                        rigth_key=['puuid', 'game_id'],
                                        how='inner')

    df_participant_perks = pd.DataFrame(df_participant_perks)
    df_game_info = pd.DataFrame(df_game_info)
    df_relation_idgame_puuid = pd.DataFrame(df_relation_idgame_puuid)

    for i in df_all_game_data.columns:
        print(i)

    table_communication = transfo.select_columns(df_all_game_data, champs=keys.vision_communication)
    table_identite_joueur = transfo.select_columns(df_all_game_data, champs=keys.identite_joueur)
    table_economie_items = transfo.select_columns(df_all_game_data, champs=keys.economie_items)
    table_cc_capacites = transfo.select_columns(df_all_game_data, champs=keys.cc_capacites)
    table_figth = transfo.select_columns(df_all_game_data, champs=keys.combat)
    table_damage = transfo.select_columns(df_all_game_data, champs=keys.degats)
    table_farming = transfo.select_columns(df_all_game_data, champs=keys.jungle_farm)
    table_objective_structures = transfo.select_columns(df_all_game_data, champs=keys.objectifs_structures)
    table_phase_de_lane = transfo.select_columns(df_all_game_data, champs=keys.phase_de_lane)
    table_soins_soutien = transfo.select_columns(df_all_game_data, champs=keys.soins_soutien)
    table_perf_globale = transfo.select_columns(df_all_game_data, champs=keys.performance_globale)

    return [df_game_info, df_participant_perks, df_relation_idgame_puuid, table_communication, 
            table_identite_joueur, table_economie_items, table_cc_capacites, table_figth, 
            table_damage, table_farming, table_objective_structures, table_phase_de_lane, 
            table_soins_soutien, table_perf_globale]

In [5]:
df_game_info, df_participant_perks, df_relation_idgame_puuid, table_communication, table_identite_joueur, table_economie_items, table_cc_capacites, table_figth, table_damage, table_farming, table_objective_structures, table_phase_de_lane, table_soins_soutien, table_perf_globale= treatgamedata()
# display(game_info)
# display(participant_data)
# display(participant_challenge)
# display(participant_perks)
# display(relation_id_puuid)
# display(participant_all_data)

{'allInPings': 0, 'assistMePings': 0, 'assists': 2, 'baronKills': 0, 'basicPings': 0, 'champExperience': 15255, 'champLevel': 16, 'championId': 39, 'championName': 'Irelia', 'championTransform': 0, 'commandPings': 3, 'consumablesPurchased': 2, 'damageDealtToBuildings': 3553, 'damageDealtToEpicMonsters': 196, 'damageDealtToObjectives': 3750, 'damageDealtToTurrets': 3553, 'damageSelfMitigated': 27429, 'dangerPings': 0, 'deaths': 8, 'detectorWardsPlaced': 0, 'doubleKills': 0, 'dragonKills': 0, 'eligibleForProgression': True, 'enemyMissingPings': 3, 'enemyVisionPings': 0, 'firstBloodAssist': False, 'firstBloodKill': False, 'firstTowerAssist': False, 'firstTowerKill': False, 'gameEndedInEarlySurrender': False, 'gameEndedInSurrender': False, 'getBackPings': 3, 'goldEarned': 10350, 'goldSpent': 9300, 'holdPings': 0, 'individualPosition': 'TOP', 'inhibitorKills': 0, 'inhibitorTakedowns': 0, 'inhibitorsLost': 4, 'item0': 3181, 'item1': 3153, 'item2': 2019, 'item3': 0, 'item4': 3047, 'item5': 20

In [6]:
display("Table cc_capacites", table_cc_capacites.shape, table_cc_capacites)
display("Table communication", table_communication.shape, table_communication)
display("Table damage", table_damage.shape, table_damage)
display("Table economie items", table_economie_items.shape, table_economie_items)
display("Table farming", table_farming.shape, table_farming)
display("Table figth", table_figth.shape, table_figth)
display("Table id joueur", table_identite_joueur.shape, table_identite_joueur)
display("Table objective structure", table_objective_structures.shape, table_objective_structures)
display("Table perf globale", table_perf_globale.shape, table_perf_globale)
display("Table phase de lane", table_phase_de_lane.shape, table_phase_de_lane)
display("Table soins soutien", table_soins_soutien.shape, table_soins_soutien)

'Table cc_capacites'

(10, 18)

,puuid,game_id,timeCCingOthers,totalTimeCCDealt,spell1Casts,spell2Casts,spell3Casts,spell4Casts,summoner1Casts,summoner1Id,summoner2Casts,summoner2Id,enemyChampionImmobilizations,abilityUses,skillshotsHit,skillshotsDodged,dodgeSkillShotsSmallWindow,landSkillShotsEarlyGame
0,qcCr6_8km5jGV3sJg05qVLecLwTol-k3dDDzhugM8ROrF5...,7799959352,10,46,234,15,47,8,3,4,6,14,12,304,18,23,1,3
1,3HTvKgEQ2zsX23fvz_dVodB4iJGa7jFbD-bwCZfC1F1qY9...,7799959352,26,185,63,0,78,8,15,11,4,4,57,149,0,22,1,0
2,uuhMlNBPDT3pLnQaGe2bXJ1CrxIuDxFD7dgcYKmdSnPL8s...,7799959352,8,66,101,24,49,6,2,4,2,12,16,180,0,38,2,0
3,dwklZfe65dWfk2ahpkvyLDdU4VmpmiKFbP9W4Iv1lhpi5j...,7799959352,0,0,177,39,27,12,6,21,5,4,0,255,64,18,1,15
4,b3jKwtQEvWhrHOKgQ7EdJCvrserlWbZ4UnRC_sXTThFsvt...,7799959352,35,211,122,70,37,9,5,7,4,4,22,238,102,17,0,22
5,ClItTtga7tOfRWWw-Np7dIKexzBEN7xRwWRC-7LAXgj_nK...,7799959352,40,124,64,23,92,9,5,4,7,14,0,188,0,64,1,0
6,l0rn2L3D6qnVcgkEm7gQQNUabxf8zW1CnOYJ5BI1Q5LbP0...,7799959352,10,489,308,94,98,5,3,4,20,11,6,505,14,21,2,1
7,xL7PH4W4vy4Oz0w16ZgXU41ilyIFkMPIRpgNyKXUE7X7BT...,7799959352,23,340,104,13,63,5,4,12,4,4,13,185,175,22,0,50
8,DkcZQVZ8faLzGb__fwTeuefkHXUidshGIUkBiv36_8vig-...,7799959352,10,238,144,61,45,9,4,4,6,21,0,259,46,44,2,7
9,IMjlDrchPi-fdzmQWEhRUcOVV4BWqJHd6t5LSZdREK40ti...,7799959352,74,300,51,54,38,8,5,4,7,14,132,151,28,58,1,5


'Table communication'

(10, 30)

...

(10, 8)

,puuid,game_id,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,laningPhaseGoldExpAdvantage,earlyLaningPhaseGoldExpAdvantage,blastConeOppositeOpponentCount,fistBumpParticipation
0,qcCr6_8km5jGV3sJg05qVLecLwTol-k3dDDzhugM8ROrF5...,7799959352,27.00,1,0,0,0,0
1,3HTvKgEQ2zsX23fvz_dVodB4iJGa7jFbD-bwCZfC1F1qY9...,7799959352,4.00,0,0,0,0,0
2,uuhMlNBPDT3pLnQaGe2bXJ1CrxIuDxFD7dgcYKmdSnPL8s...,7799959352,1.00,1,0,0,0,0
3,dwklZfe65dWfk2ahpkvyLDdU4VmpmiKFbP9W4Iv1lhpi5j...,7799959352,0.00,1,0,0,0,0
4,b3jKwtQEvWhrHOKgQ7EdJCvrserlWbZ4UnRC_sXTThFsvt...,7799959352,21.75,1,0,0,0,0
5,ClItTtga7tOfRWWw-Np7dIKexzBEN7xRwWRC-7LAXgj_nK...,7799959352,5.00,2,0,0,0,1
6,l0rn2L3D6qnVcgkEm7gQQNUabxf8zW1CnOYJ5BI1Q5LbP0...,7799959352,153.00,5,0,0,0,4
7,xL7PH4W4vy4Oz0w16ZgXU41ilyIFkMPIRpgNyKXUE7X7BT...,7799959352,105.00,3,1,0,0,0
8,DkcZQVZ8faLzGb__fwTeuefkHXUidshGIUkBiv36_8vig-...,7799959352,68.00,2,1,0,0,3
9,IMjlDrchPi-fdzmQWEhRUcOVV4BWqJHd6t5LSZdREK40ti...,7799959352,10.00,2,0,0,0,2


'Table soins soutien'

(10, 10)

,puuid,game_id,totalHeal,totalHealsOnTeammates,totalDamageShieldedOnTeammates,totalUnitsHealed,effectiveHealAndShielding,saveAllyFromDeath,killsOnRecentlyHealedByAramPack,quickCleanse
0,qcCr6_8km5jGV3sJg05qVLecLwTol-k3dDDzhugM8ROrF5...,7799959352,8456,0,0,1,0.000000,0,0,0
1,3HTvKgEQ2zsX23fvz_dVodB4iJGa7jFbD-bwCZfC1F1qY9...,7799959352,4968,0,0,1,0.000000,0,0,0
2,uuhMlNBPDT3pLnQaGe2bXJ1CrxIuDxFD7dgcYKmdSnPL8s...,7799959352,3248,0,0,1,0.000000,0,0,0
3,dwklZfe65dWfk2ahpkvyLDdU4VmpmiKFbP9W4Iv1lhpi5j...,7799959352,3776,0,0,1,0.000000,0,0,0
4,b3jKwtQEvWhrHOKgQ7EdJCvrserlWbZ4UnRC_sXTThFsvt...,7799959352,2754,184,0,2,184.000000,0,0,0
5,ClItTtga7tOfRWWw-Np7dIKexzBEN7xRwWRC-7LAXgj_nK...,7799959352,3500,0,0,1,0.000000,0,0,0
6,l0rn2L3D6qnVcgkEm7gQQNUabxf8zW1CnOYJ5BI1Q5LbP0...,7799959352,12760,0,0,1,0.000000,0,0,0
7,xL7PH4W4vy4Oz0w16ZgXU41ilyIFkMPIRpgNyKXUE7X7BT...,7799959352,3181,0,0,1,0.000000,0,0,0
8,DkcZQVZ8faLzGb__fwTeuefkHXUidshGIUkBiv36_8vig-...,7799959352,7384,0,0,1,0.000000,0,0,0
9,IMjlDrchPi-fdzmQWEhRUcOVV4BWqJHd6t5LSZdREK40ti...,7799959352,6079,0,664,1,664.104797,0,0,0


In [ ]:
def get_max_length(col : pd.Series):
    x = 0
    for i in col.astype(str):
        if len(i) > x:
            x = len(i)
    return x

def create_schema(table : pd.DataFrame, table_name : str):
    type_map = {
        "int64": "INTEGER",
        "float64": "REAL",
        "object": "VARCHAR",
        "bool": "BOOLEAN",
        "datetime64[ns]": "TIMESTAMP",
        "str": "VARCHAR"
    }

    columns = []
    for col in table.columns:
        col_type = type_map.get(table[col].dtype.name, "TEXT")
        columns.append(f"{col} {col_type}")

    txt = f"CREATE TABLE {table_name} (\n    " + ",\n    ".join(columns) + ","+ f"\n\tCONSTRAINT pk_{table_name}\n\t\tPRIMARY KEY (puuid, game_id)\n);"
    return txt

tables = {
    "communication":        table_communication,
    "cc_capacites":         table_cc_capacites,
    "damage":               table_damage,
    "economie_items":       table_economie_items,
    "farming":              table_farming,
    "fight":                table_figth,
    "identite_joueur":      table_identite_joueur,
    "objective_structures": table_objective_structures,
    "perf_globale":         table_perf_globale,
    "phase_de_lane":        table_phase_de_lane,
    "soins_soutien":        table_soins_soutien,
}

schemas = [create_schema(df, name) for name, df in tables.items()]

with open(r"src_sql\schemas.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(schemas))

In [ ]:
comm = ['pings', 'vision', 'wards', 'ping', 'ward']

# def select_columns(df : pd.DataFrame, keys_words : list[str]) -> pd.DataFrame:
#     """Capture les colonnes en fonction de mots clés"""
#     base_ids = ['puuid', 'game_id']
#     keys_words += base_ids
#     selected_cols = sorted([x for x in df.columns if any(keyword in x.lower() for keyword in keys_words)])
#     df_new = df[selected_cols]
#     print([s for s in selected_cols if s not in base_ids])
#     return df_new

def select_columns(df_base : pd.DataFrame, champs : list[str]) -> pd.DataFrame:
    """Capture les colonnes en fonction d'une liste"""
    df_new = df_base[champs]
    return df_new

table_communication = select_columns(participant_all_data, keys_words=comm)

In [ ]:
teams = read_json(r"C:\Users\najim\Documents\Projets\LeagueOfLegends\data\raw\game_data.json")
teams = teams['info']['teams']
print(teams[0]['bans'][0])

def parse_teams(teams_data):

    bans = []

    for i in teams:
        current_bans = i['bans']
        current_id = i['teamId']
        win = i['win']
        for x in current_bans:
            x.update({'team_id': current_id, 'win': win})
            x.pop('pickTurn')
        bans += current_bans
        print(bans)

parse_teams(teams)